### Load and parse data

In [31]:
import numpy as np
from data_io import load_data
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import pandas as pd

In [3]:
# Load files
files = [
    './inputs/Location1_train.csv', 
    './inputs/Location2_train.csv', 
    './inputs/Location3_train.csv', 
    './inputs/Location4_train.csv'
]

data = load_data(files)
    
data.describe()


Combined DataFrame Shape: (140160, 10)

First 5 rows:
   temperature_2m  relativehumidity_2m  dewpoint_2m  windspeed_10m  \
0      271.205556                   85   268.983333           1.44   
1      271.150000                   86   269.094444           2.06   
2      270.261111                   91   268.983333           1.30   
3      270.594444                   88   268.872222           1.30   
4      270.538889                   88   268.761111           2.47   

   windspeed_100m  winddirection_10m  winddirection_100m  windgusts_10m  \
0            1.26                146                 162            1.4   
1            3.99                151                 158            4.4   
2            2.78                148                 150            3.2   
3            2.69                 58                 105            1.6   
4            4.43                 58                  84            4.0   

    Power   Location  
0  0.1635  Location1  
1  0.1424  Location1  
2  0.

,temperature_2m,relativehumidity_2m,dewpoint_2m,windspeed_10m,windspeed_100m,winddirection_10m,winddirection_100m,windgusts_10m,Power
count,140160.000000,140160.000000,140160.000000,140160.000000,140160.000000,140160.000000,140160.000000,140160.000000,140160.000000
mean,281.001423,70.756307,275.535907,4.187559,6.916478,201.632734,201.591695,8.091158,0.312441
std,12.183641,17.000203,11.655400,2.027149,3.056636,100.079917,101.105919,3.626641,0.253774
min,238.038889,9.000000,235.261111,0.000000,0.000000,1.000000,0.000000,0.500000,0.000000
25%,271.927778,58.000000,267.372222,2.650000,4.740000,129.000000,128.000000,5.300000,0.099700
50%,280.983333,73.000000,275.427778,3.890000,6.710000,212.000000,212.000000,7.700000,0.246900
75%,291.372222,85.000000,285.538889,5.410000,8.840000,288.000000,290.000000,10.300000,0.486400
max,307.983333,100.000000,299.094444,18.530000,24.590000,360.000000,360.000000,29.000000,0.988800


In [ ]:
# 1. Check for abnormal values
ranges = {
'temperature_2m': (230, 320),
'dewpoint_2m': (230, 320),
'relativehumidity_2m': (0, 100),
'windspeed_10m': (0, 50),
'windspeed_100m': (0, 50),
'wdcos_10': (0, 1),
'wdsin_10': (0, 1),
'wdcos_100': (0, 1),
'wdsin_10': (0, 1),
'Power': (0, 1)
}

# 2. Iterate and apply masks
for col, (low, high) in ranges.items():
    if col in data.columns:
        if low is not None:
            data[col].mask(data[col] < low, np.nan, inplace=True)
        if high is not None:
            data[col].mask(data[col] > high, np.nan, inplace=True)

data.describe()

C:\Users\cfmoralesruval\AppData\Local\Temp\ipykernel_2528\325471989.py:17: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  data[col].mask(data[col] < low, np.nan, inplace=True)
C:\Users\cfmoralesruval\AppData\Local\Temp\ipykernel_2528\325471989.py:19: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series throu

,temperature_2m,relativehumidity_2m,dewpoint_2m,windspeed_10m,windspeed_100m,winddirection_10m,winddirection_100m,windgusts_10m,Power
count,140160.000000,140160.000000,140160.000000,140160.000000,140160.000000,140160.000000,140160.000000,140160.000000,140160.000000
mean,281.001423,70.756307,275.535907,4.187559,6.916478,201.632734,201.591695,8.091158,0.312441
std,12.183641,17.000203,11.655400,2.027149,3.056636,100.079917,101.105919,3.626641,0.253774
min,238.038889,9.000000,235.261111,0.000000,0.000000,1.000000,0.000000,0.500000,0.000000
25%,271.927778,58.000000,267.372222,2.650000,4.740000,129.000000,128.000000,5.300000,0.099700
50%,280.983333,73.000000,275.427778,3.890000,6.710000,212.000000,212.000000,7.700000,0.246900
75%,291.372222,85.000000,285.538889,5.410000,8.840000,288.000000,290.000000,10.300000,0.486400
max,307.983333,100.000000,299.094444,18.530000,24.590000,360.000000,360.000000,29.000000,0.988800


In [18]:
# Separate data into target and features
y = data.Power
X = data.drop(["Power"], axis=1)

In [ ]:
# Encode the Location
X_transformed = pd.get_dummies(X, columns=['Location'], dtype=int)

np.shape(X_transformed)
X_transformed




,temperature_2m,relativehumidity_2m,dewpoint_2m,windspeed_10m,windspeed_100m,winddirection_10m,winddirection_100m,windgusts_10m,Location_Location1,Location_Location2,Location_Location3,Location_Location4
0,271.205556,85,268.983333,1.44,1.26,146,162,1.4,1,0,0,0
1,271.150000,86,269.094444,2.06,3.99,151,158,4.4,1,0,0,0
2,270.261111,91,268.983333,1.30,2.78,148,150,3.2,1,0,0,0
3,270.594444,88,268.872222,1.30,2.69,58,105,1.6,1,0,0,0
4,270.538889,88,268.761111,2.47,4.43,58,84,4.0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
140155,263.816667,82,261.261111,1.61,1.86,150,216,3.3,0,0,0,1
140156,260.150000,89,258.705556,2.94,4.22,162,175,3.8,0,0,0,1
140157,260.205556,89,258.761111,2.60,5.75,164,167,4.1,0,0,0,1
140158,260.872222,86,259.038889,3.18,7.53,167,169,5.4,0,0,0,1
